### 1. Install Dependensi
Jalankan cell ini untuk menginstal semua library yang dibutuhkan untuk pemrosesan audio, model XTTS, server FastAPI, dan interface Gradio.

In [2]:
!pip install coqui-tts

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.2/850.2 kB 10.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.7/997.7 kB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.9/682.9 kB 11.5 MB/s  0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=5afe29f52804096ef364d5f748d1b030d2afaad060c3512b9e95a0f731f95a62
  Stored in directory: /root/.cache/pip/wheels/0b/1d/03/175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
Successfully built docopt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [coqui-tts]


### 2. Pre-load Model ke GPU
Kita memuat model XTTS-v2 ke dalam memory GPU (`cuda`) terlebih dahulu sebelum menyalakan server. Langkah ini menjamin tidak akan ada timeout atau error 502 akibat proses inisialisasi model yang lambat saat request pertama masuk.

In [3]:
import torch
import os
from TTS.api import TTS

# Deteksi device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan device: {device.upper()}")

# Unduh & inisialisasi model ke memory sebelum server berjalan
print("⏳ Memulai pemuatan model XTTS-v2 ke memory (ini memerlukan waktu beberapa menit)...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("🎉 MODEL XTTS-V2 BERHASIL DIMUAT KE MEMORY!")

Menggunakan device: CUDA
⏳ Memulai pemuatan model XTTS-v2 ke memory (ini memerlukan waktu beberapa menit)...
You must confirm the following:
  "I have purchased a commercial license from Coqui: licensing@coqui.ai"
  "Otherwise, I agree to the terms of the non-commercial CPML: https://tts-hub.github.io/cpml" - [y/n]
   > y


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.


🎉 MODEL XTTS-V2 BERHASIL DIMUAT KE MEMORY!


### Panduan Penyiapan File Referensi Suara (Clone Voice):
1. Klik ikon folder 📁 di panel kiri Google Colab.
2. Seret (drag & drop) file audio referensi Anda (format `.wav`, durasi ideal 5-10 detik, bersih tanpa noise) ke panel tersebut.
3. Beri nama file tersebut `speaker.wav` (atau sesuaikan jalurnya nanti).

---
## OPSI I: Jalankan dengan Gradio (Rekomendasi Cepat & Stabil)
Gradio akan otomatis memberikan public link aktif berakhiran `.gradio.live` secara gratis tanpa setup tunnel tambahan.

In [4]:
# import gradio as gr

# def generate_tts_gradio(text, language, speaker_audio):
#     output_path = "output_gradio.wav"
#     # Memproses TTS menggunakan model yang sudah di-load
#     tts.tts_to_file(
#         text=text,
#         speaker_wav=speaker_audio if speaker_audio else "speaker.wav",
#         language=language,
#         file_path=output_path
#     )
#     return output_path

# interface = gr.Interface(
#     fn=generate_tts_gradio,
#     inputs=[
#         gr.Textbox(label="Teks untuk disintesis", value="Halo, ini adalah suara hasil kloning menggunakan model XTTS-v2 di Google Colab!"),
#         gr.Dropdown(label="Bahasa", choices=["id", "en", "es", "fr", "de", "it", "pt", "ja", "zh-cn"], value="id"),
#         gr.Audio(label="Suara Referensi (Kloning)", type="filepath")
#     ],
#     outputs=gr.Audio(label="Hasil Sintesis Suara", type="filepath"),
#     title="XTTS-v2 Voice Cloning Server"
# )

# # Menjalankan Gradio dengan link public share aktif
# interface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ef0cca0075988daf75.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ef0cca0075988daf75.gradio.live


---
## OPSI II: Jalankan dengan FastAPI + LocalTunnel (Untuk API Integration)
Jika aplikasi frontend Anda membutuhkan response JSON terstruktur/API backend, gunakan opsi ini.

In [11]:
%%writefile app.py
import os
import torch
import librosa
import soundfile as sf
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field
from TTS.api import TTS

# ==========================================
# KONFIGURASI AUDIO POST-PROCESSING (GLOBAL)
# ==========================================
# Batas aman pitch_shift: -2.0 s/d 2.5 (semakin rendah = suara semakin berat/bass, semakin tinggi = suara cempreng/chipmunk)
DEFAULT_PITCH_SHIFT = 1
# Batas aman speed_rate: 0.9 s/d 1.1 (mengatur kecepatan bicara tanpa merubah pitch suara)
DEFAULT_SPEED_RATE = 1.0

app = FastAPI(title="XTTS Audio Engineering API")

# Memanfaatkan model yang sudah ter-cache ke GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

def apply_audio_effects(input_path: str, output_path: str, pitch_shift: float, speed_rate: float):
    try:
        # Load audio mentah dari XTTS
        y, sr = librosa.load(input_path, sr=None)

        # 1. Terapkan Pitch Shifting jika nilainya tidak 0.0
        if pitch_shift != 0.0:
            y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch_shift)

        # 2. Terapkan Time Stretching (Speed Rate) jika nilainya tidak 1.0
        if speed_rate != 1.0 and speed_rate > 0.0:
            y = librosa.effects.time_stretch(y, rate=speed_rate)

        # Simpan kembali hasil modifikasi audio
        sf.write(output_path, y, sr)
    except Exception as e:
        raise RuntimeError(f"Gagal melakukan post-processing audio: {str(e)}")

class TTSRequest(BaseModel):
    text: str
    language: str = "en"
    speaker_wav: str = "speaker.wav"
    pitch_shift: float = Field(default=DEFAULT_PITCH_SHIFT, ge=-2.0, le=2.5, description="Geser nada suara. Batas aman: -2.0 hingga 2.5")
    speed_rate: float = Field(default=DEFAULT_SPEED_RATE, ge=0.9, le=1.1, description="Kecepatan bicara. Batas aman: 0.9 hingga 1.1")

@app.post("/generate-audio/")
async def generate_audio(payload: TTSRequest):
    if not os.path.exists(payload.speaker_wav):
        raise HTTPException(status_code=400, detail=f"File referensi audio '{payload.speaker_wav}' tidak ditemukan di runtime Colab.")

    raw_temp_path = "raw_temp_output.wav"
    processed_output_path = "processed_output.wav"

    try:
        # Langkah 1: Generate audio mentah menggunakan XTTS
        tts_model.tts_to_file(
            text=payload.text,
            speaker_wav=payload.speaker_wav,
            language=payload.language,
            file_path=raw_temp_path
        )

        # Langkah 2: Proses audio menggunakan Librosa & Soundfile
        apply_audio_effects(
            input_path=raw_temp_path,
            output_path=processed_output_path,
            pitch_shift=payload.pitch_shift,
            speed_rate=payload.speed_rate
        )

        # Langkah 3: Kembalikan respon file audio final
        return FileResponse(processed_output_path, media_type="audio/wav", filename="synthesized_audio.wav")

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Gagal memproses audio: {str(e)}")
    finally:
        # Bersihkan file temporer untuk menghemat penyimpanan
        if os.path.exists(raw_temp_path):
            os.remove(raw_temp_path)

Overwriting app.py


Jalankan server FastAPI di background:

In [12]:
import subprocess
# Jalankan Uvicorn di background
subprocess.Popen(["uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000", "--timeout-keep-alive", "600"])

<Popen: returncode: None args: ['uvicorn', 'app:app', '--host', '127.0.0.1',...>

Expose port FastAPI Anda ke internet menggunakan LocalTunnel gratis:

In [ ]:
# Install LocalTunnel secara global
!npm install -g localtunnel

# Dapatkan IP Publik Colab Anda (diperlukan sebagai password keamanan LocalTunnel)
import urllib.request
print("Password untuk LocalTunnel Anda adalah:")
print(urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip())
print("------------------------------------------")

# Expose port 8000
!lt --port 8000

⠙⠹⠸⠼⠴
changed 22 packages in 700ms
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴Password untuk LocalTunnel Anda adalah:
34.6.165.103
------------------------------------------
your url is: https://fresh-cars-trade.loca.lt


### 💡 Tips Anti-Disconnect Google Colab:
Agar tab Google Colab Anda tidak terputus karena terdeteksi tidak aktif saat ditinggal hackathon, tekan `Ctrl+Shift+I` (atau `Cmd+Option+I` di Mac) untuk membuka Console Browser Anda, lalu tempel (paste) kode Javascript berikut dan tekan **Enter**:

```javascript
function KeepAlive(){
  console.log("Mengirim sinyal agar Colab tetap aktif...");
  document.querySelector("colab-connect-button").click();
}
setInterval(KeepAlive, 60000);
```

# Sonora TTS — Demo Server di Colab (GRATIS)

Jalankan cell 1–4 berurutan. Dapat URL publik → pasang ke Vercel env `TTS_URL` → redeploy Sonora → demo.
Catatan: runtime Colab mati kalau idle ±90 menit / tab ditutup. Untuk demo 1–2 jam aman.

In [7]:
# 1) Install dependency (sekali per session)
# !pip install -q fastapi "uvicorn[standard]" edge-tts pydantic

KeyboardInterrupt: 

In [ ]:
# # 2) Tulis server TTS
# %%writefile server.py
# import base64, os, torch, shutil, urllib.request, asyncio
# from fastapi import FastAPI, Header, HTTPException
# from pydantic import BaseModel, Field
# from TTS.api import TTS
# from google.colab import drive

# # Fungsi pembantu sambungkan Drive
# def sambungkan_drive(force=False):
#     if force or not os.path.exists('/content/drive'):
#         print("Menghubungkan ulang Google Drive...")
#         try:
#             drive.mount('/content/drive', force_remount=True)
#         except Exception as e:
#             print(f"Gagal menyambungkan drive: {e}")

# sambungkan_drive()

# device = "cuda" if torch.cuda.is_available() else "cpu"
# PATH_MODEL_DRIVE = "/content/drive/MyDrive/xtts_finetune_output/run/training/GPT_XTTS_FT-October-28-2026_12+12PM-0000000/best_model.pth"
# PATH_CONFIG_DRIVE = "/content/drive/MyDrive/xtts_finetune_output/run/training/GPT_XTTS_FT-October-28-2026_12+12PM-0000000/config.json"
# PATH_VOCAB_DRIVE = "/content/drive/MyDrive/xtts_finetune_output/vocab.json"
# temp_model_dir = "/content/temp_xtts_model"
# local_config_path = os.path.join(temp_model_dir, "config.json")
# local_model_path = os.path.join(temp_model_dir, "model.pth")
# local_vocab_path = os.path.join(temp_model_dir, "vocab.json")

# # Inisialisasi placeholder model XTTS
# tts_model = None
# model_loading_status = "not_started"

# def load_tts_model():
#     global tts_model, model_loading_status
#     model_loading_status = "loading"
#     try:
#         os.makedirs(temp_model_dir, exist_ok=True)
#         if os.path.exists(PATH_MODEL_DRIVE) and os.path.exists(PATH_CONFIG_DRIVE):
#             shutil.copy2(PATH_CONFIG_DRIVE, local_config_path)
#             shutil.copy2(PATH_MODEL_DRIVE, local_model_path)
#             if os.path.exists(PATH_VOCAB_DRIVE):
#                 shutil.copy2(PATH_VOCAB_DRIVE, local_vocab_path)
#             else:
#                 vocab_url = "https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json"
#                 urllib.request.urlretrieve(vocab_url, local_vocab_path)

#             tts_model = TTS(model_path=temp_model_dir, config_path=local_config_path).to(device)
#             print("🎉 Model CUSTOM Anda berhasil dimuat!")
#             model_loading_status = "ready"
#             return
#     except Exception as e:
#         print(f"⚠️ Gagal memuat model custom: {e}. Menggunakan fallback...")

#     if tts_model is None:
#         try:
#             tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
#             model_loading_status = "ready"
#         except Exception as fallback_err:
#             print(f"❌ Gagal memuat fallback model: {fallback_err}")
#             model_loading_status = "failed"

# # Siapkan file referensi audio
# reference_audio = "reference_sample.wav"
# if not os.path.exists(reference_audio):
#     urllib.request.urlretrieve(
#         "https://raw.githubusercontent.com/coqui-ai/TTS/dev/tests/data/ljspeech/wavs/LJ001-0001.wav",
#         reference_audio,
#     )

# app = FastAPI(title="Sonora XTTS")
# API_KEY = os.environ.get("SONORA_TTS_KEY", "LLM_2178534706395686_2kIJ3CzbX_dZFWspPkIgwfRGWbQ")

# @app.on_event("startup")
# async def startup_event():
#     # Jalankan proses loading model secara background agar FastAPI bisa langsung aktif melayani request awal (health check)
#     loop = asyncio.get_running_loop()
#     loop.run_in_executor(None, load_tts_model)

# class SpeakRequest(BaseModel):
#     text: str = Field(min_length=1, max_length=2000)
#     language: str = "en"
#     voice: str = ""

# @app.get("/health")
# def health():
#     return {
#         "ok": True,
#         "model_status": model_loading_status,
#         "device": device
#     }

# @app.post("/speak")
# async def speak(req: SpeakRequest, x_sonora_key: str = Header(default="")):
#     if API_KEY and x_sonora_key != API_KEY:
#         raise HTTPException(status_code=401, detail="Unauthorized")

#     if tts_model is None:
#         raise HTTPException(status_code=503, detail=f"Model is still loading or failed to load. Current status: {model_loading_status}")

#     speaker_wav = req.voice if req.voice and os.path.exists(req.voice) else reference_audio
#     output_path = "/content/output_temp.wav"

#     try:
#         # Jalankan sintesis di threadpool agar tidak memblokir event loop FastAPI
#         loop = asyncio.get_running_loop()
#         await loop.run_in_executor(None, lambda: tts_model.tts_to_file(
#             text=req.text,
#             speaker_wav=speaker_wav,
#             language=req.language,
#             file_path=output_path
#         ))

#         with open(output_path, "rb") as f:
#             audio_data = f.read()
#     except Exception as e:
#         raise HTTPException(status_code=502, detail=f"TTS XTTS failed: {e}")
#     finally:
#         if os.path.exists(output_path):
#             os.remove(output_path)

#     if not audio_data:
#         raise HTTPException(status_code=502, detail="TTS returned empty audio")
#     return {"audio": "data:audio/wav;base64," + base64.b64encode(audio_data).decode()}

In [ ]:
# # 3) Jalankan server di background + tes lokal
# import os
# os.environ["SONORA_TTS_KEY"] = "LLM_2178534706395686_2kIJ3CzbX_dZFWspPkIgwfRGWbQ"  # ganti sesukamu, samakan dengan TTS_KEY di Vercel
# !nohup uvicorn server:app --host 127.0.0.1 --port 7860 > server.log 2>&1 &
# import time; time.sleep(4)
# !curl -s http://127.0.0.1:7860/health

In [ ]:
# # 4) Expose ke internet + tampilkan URL publik otomatis
# # Colab TIDAK punya IP publik, jadi kita pakai tunnel Cloudflare.
# # Tunnel ini yang memberi kamu hostname publik (bukan IP Colab).
# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
# import re, subprocess, threading, time
# proc = subprocess.Popen(
#     ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
# )
# public_url = None
# def _watch():
#     global public_url
#     for line in proc.stdout:
#         m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
#         if m and not public_url:
#             public_url = m.group(0)
#             print("\n✅ PUBLIC URL:", public_url)
#             print("👉 TTS_URL =", public_url + "/speak")
# threading.Thread(target=_watch, daemon=True).start()
# time.sleep(15)
# if not public_url:
#     print("URL belum muncul — tunggu 10 detik lalu run ulang cell ini SAJA.")
# else:
#     import urllib.request
#     print("Health check:", urllib.request.urlopen(public_url + "/health", timeout=30).read().decode())

In [ ]:
# # Tampilkan log server untuk melihat error atau progress load model
# with open('server.log', 'r') as f:
#     print(f.read())

In [ ]:
# import urllib.request, json, time

# print("Memantau status inisialisasi model di server lokal...")
# for i in range(10):
#     try:
#         response = urllib.request.urlopen("http://127.0.0.1:7860/health", timeout=2)
#         data = json.loads(response.read().decode())
#         status = data.get("model_status", "unknown")
#         print(f"[Detik {(i+1)*3}] Status Model: {status.upper()}")
#         if status == "ready":
#             print("\n🎉 Model SIAP! Anda sudah bisa mengirimkan request suara tanpa khawatir timeout.")
#             break
#     except Exception as e:
#         print(f"[Detik {(i+1)*3}] Server belum merespons port (sedang booting)... ")
#     time.sleep(3)

## Setelah dapat URL

1. Copy URL `https://....trycloudflare.com` dari output cell 4.
2. Di Vercel → Project Sonora → Settings → Environment Variables:
   - `TTS_URL` = `https://....trycloudflare.com/speak`
   - `TTS_KEY` = `demo-key-123` (sama seperti di cell 3)
3. Redeploy Sonora.
4. Test: upload artwork → Generate → Play narration.

URL quick tunnel **berubah tiap Colab restart**. Untuk URL permanen gratis, pakai Render (lihat `tts-space/README.md`).